In [2]:
import pandas as pd
import numpy as np

# Virginia approximate bounds
va_lat_min, va_lat_max = 36.5, 39.5
va_lon_min, va_lon_max = -83.7, -75.0

# Generate points inside Virginia
n_inside = 90
inside_latitudes = np.random.uniform(va_lat_min, va_lat_max, n_inside)
inside_longitudes = np.random.uniform(va_lon_min, va_lon_max, n_inside)

# Outside points (e.g., California, Florida, or Atlantic Ocean)
outside_coords = [
    (34.0522, -118.2437),  # Los Angeles, CA
    (40.7128, -74.0060),   # NYC
    (25.7617, -80.1918),   # Miami, FL
    (41.8781, -87.6298),   # Chicago, IL
    (30.2672, -97.7431),   # Austin, TX
    (35.6895, 139.6917),   # Tokyo
    (51.5074, -0.1278),    # London
    (0.0, 0.0),            # Null Island
    (-33.8688, 151.2093),  # Sydney, AU
    (55.7558, 37.6173)     # Moscow
]

# Split into separate lists
outside_latitudes, outside_longitudes = zip(*outside_coords)

# Combine inside and outside data
latitudes = np.concatenate([inside_latitudes, outside_latitudes])
longitudes = np.concatenate([inside_longitudes, outside_longitudes])

# Labels: "inside" for the first 90, "outside" for the rest
labels = ['inside'] * n_inside + ['outside'] * len(outside_coords)

# Create DataFrame
df = pd.DataFrame({
    'latitude': latitudes,
    'longitude': longitudes,
    'location_type': labels
})

# Show first few rows
print(df.head())



    latitude  longitude location_type
0  38.221780 -78.598248        inside
1  37.848903 -80.726237        inside
2  37.017697 -75.706331        inside
3  39.274016 -80.241796        inside
4  39.131245 -78.167715        inside


In [5]:
# Libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# Load Virginia shapefile

va_shapefile = "./datasets/cb_2018_us_state_20m/cb_2018_us_state_20m.shp"
virginia = gpd.read_file(va_shapefile)

print("✅ Loaded Virginia shapefile")
print("Virginia CRS:", virginia.crs)

#  Load your latitude/longitude dataset 
csv_path = "path/to/your/coordinates.csv"
points_df = df

# Create geometry column from lon/lat
geometry = [Point(xy) for xy in zip(points_df.longitude, points_df.latitude)]
points_gdf = gpd.GeoDataFrame(points_df, geometry=geometry)

# Set CRS to WGS84 (EPSG:4326) if not already set
if points_gdf.crs is None:
    points_gdf.set_crs(epsg=4326, inplace=True)
    print("✅ Set CRS for points to EPSG:4326 (WGS84)")

print("Points CRS:", points_gdf.crs)

# Check and align CRS 
if points_gdf.crs != virginia.crs:
    print("⚠️ CRS mismatch detected. Reprojecting points to match Virginia shapefile...")
    points_gdf = points_gdf.to_crs(virginia.crs)
    print("Points reprojected. CRS is now:", points_gdf.crs)
else:
    print("CRS match. No reprojection needed.")

#  Spatial join to find which points are within Virginia
joined = gpd.sjoin(points_gdf, virginia, how="left", predicate='within')

# Points outside Virginia will have NaN in 'index_right' (no match found)
outside_va = joined[joined['index_right'].isna()]

# Result 
print(" Points OUTSIDE Virginia:")
print(outside_va[['longitude', 'latitude']].head())


✅ Loaded Virginia shapefile
Virginia CRS: EPSG:4269
✅ Set CRS for points to EPSG:4326 (WGS84)
Points CRS: EPSG:4326
⚠️ CRS mismatch detected. Reprojecting points to match Virginia shapefile...
Points reprojected. CRS is now: EPSG:4269
 Points OUTSIDE Virginia:
    longitude   latitude
2  -75.706331  37.017697
54 -76.200591  38.048192
60 -75.033116  38.204420
61 -76.217724  37.294164
75 -75.645000  36.607160
